In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import re
from google.colab import files  # Import for Colab upload widget

# ==========================================
# 1. CONFIGURATION
# ==========================================

MERGE_CONFIG = [
    ('respondents_role', 'respondents_role_others'),
    ('comorbidities', 'comorbidities_others'),
    ('repetitive_behaviours', 'repetitive_behaviours_others'),
    ('technology_used', 'technology_used_others'),
    ('purpose_technology_used', 'purpose_technology_used_others'),
    ('child_safety_solution_used', 'child_safety_solution_used_others'),
    ('reason_belief', 'reason_no_belief_ai'),
    ('best_information_capture_means', 'best_information_capture_means_others'),
    ('device_feedback_expectation', 'device_feedback_expectation_others')
]

COLUMNS_TO_DROP = [
    'Response ID',
    'consent_to_study'

]

RENAME_CONFIG = {
    '1. By participating, you confirm you are 18 or older and consent to contribute to this study.I am the parent/guardian or caregiver/teacher of a child with autism (ASD) aged between 06 to 16 years of age.': 'consent_to_study',
    '2. What is the relationship between you and the child with ASD?': 'respondents_role',
    '2. What is the relationship between you and the child with ASD? - Other (please specify)': 'respondents_role_others',
    '3. How old is the child?': 'childs_age',
    '4. What is the gender of the child?': 'childs_gender',
    '5. What is the level of autism that the child has?': 'childs_autism_severity',
    '6. Does anyone else in the family have autism/ASD?': 'autism_history_family',
    '7. At what age was autism diagnosed/detected?': 'age_autism_diagnosed',
    '8. Does the child suffer from comorbidity?': 'comorbidities',
    '8. Does the child suffer from comorbidity? - Other (please specify)': 'comorbidities_others',
    '9. Does the child with ASD interact with others without prompts?': 'social_interaction_status',
    '10. Does the individual with ASD express any emotional attachment to family or friends?': 'emotional_attachment_status',
    '11. Does your child maintain eye contact while interacting with others?': 'maintain_eye_contact',
    '12. What is the frequent style of communication by the individual with ASD?': 'favoured_communication_style',
    '13. Does the child with ASD speak\xa0with unusual volume, tone, rhythm, or rate of speech?': 'unusual_verbal_behaviour',
    '14. What are (is) the common repetitive behaviour of the child with ASD?': 'repetitive_behaviours',
    '14. What are (is) the common repetitive behaviour of the child with ASD? - Other (please specify)': 'repetitive_behaviours_others',
    '15. Does the child undergo education in a special education school, individual counselling, therapy sessions? If Yes, please list here.': 'special_education_need',
    '16. Do you use any technological aid for the child?\xa0 ': 'tech_user_status',
    '17. If "Yes in Q16": What are the devices/technologies that are used?': 'technology_used',
    '17. If "Yes in Q16": What are the devices/technologies that are used? - Other (please specify)':'technology_used_others',
    '18. If "Yes in Q16": What is the purpose for which the technology/device is used?': 'purpose_technology_used',
    '18. If "Yes in Q16": What is the purpose for which the technology/device is used? - Other (please specify)':'purpose_technology_used_others',
    '19. Do you use any solutions specifically for child safety?': 'child_safety_solution_used',
    '19. Do you use any solutions specifically for child safety? - Other (please specify)':'child_safety_solution_used_others',
    '20. Do you believe that AI embedded devices may prove helpful in improving any of the issues that children with ASD has?': 'belief_in_ai_use',
    '21. If "Yes", how do you thing AI or advanced technology should help child or parents/career etc.? (300 Characters)':'reason_belief',
    '22. If "No", Wy do you believe that AI embedded devices may not be helpful?': 'reason_no_belief_ai',
    '23. What do you believe is the best way to capture information from children with ASD using technology?': 'best_information_capture_means',
    '23. What do you believe is the best way to capture information from children with ASD using technology? - Other (please specify)':'best_information_capture_means_others',
    '24. If the child was to use a AI/wearable device; What are your expectations from the communication and support provided by AI/wearable device?': 'device_feedback_expectation',
    '24. If the child was to use a AI/wearable device; What are your expectations from the communication and support provided by AI/wearable device? - Other (please specify)':'device_feedback_expectation_others',
    '25. Does your child show aggressive behaviour when the child is unable to communicate their needs to you?': 'aggressive_behaviour',
    '26. Has the child inflicted injury to self due to inability to communicate their needs with you or caregiver?': 'self_injury_incidents',
    '27. Does the child show\xa0 intolerance towards uncertainty or changes in their environment? (new toys, strangers, new voices etc)':'intolerance_to_uncertainty',
    '28. What is most challenging aspect of dealing with the child (e.g. Food, toilet, mobility, anger outbursts, etc)? How do you deal with the situation when the challenge arise ?Challenge:': 'most_challenging_aspect',
    '29. Solution:': 'solution_most_challenging_aspect',
}

# ==========================================
# 2. PROCESSING FUNCTIONS
# ==========================================

def load_data_from_upload():
    """
    Triggers Colab upload widget, loads the first file uploaded,
    and returns the DataFrame and the filename.
    """
    print("Please upload your dataset (CSV or Excel)...")
    uploaded = files.upload()

    if not uploaded:
        print("[ERROR] No file uploaded.")
        return None, None

    # Get the first filename from the uploaded dictionary
    file_path = list(uploaded.keys())[0]
    print(f"[INFO] File uploaded: {file_path}")

    try:
        if file_path.endswith('.csv'):
            try:
                df = pd.read_csv(file_path)
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1')
        elif file_path.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(file_path)
        else:
            print("[ERROR] Unsupported file format. Please upload CSV or Excel.")
            return None, None

        print(f"[SUCCESS] Data loaded. Shape: {df.shape}")
        return df, file_path
    except Exception as e:
        print(f"[ERROR] Failed to load data: {e}")
        return None, None

def format_column_names(df):
    if df is None: return None
    df = df.copy()
    def clean_col(col_name):
        return (str(col_name).strip()
                .replace("'", "")
                .replace("\xa0", " ")
                .replace("\t", " ")
                .replace(" ", "_")
                .lower())
    df.columns = [clean_col(c) for c in df.columns]
    print("[INFO] Column names formatted.")
    return df

def rename_specific_columns(df, rename_map=None):
    if df is None: return None
    df = df.copy()
    if rename_map:
        cleaned_rename_map = {}
        for original_key, new_val in rename_map.items():
            formatted_key = (original_key.strip()
                             .replace("'", "")
                             .replace("\xa0", " ")
                             .replace("\t", " ")
                             .replace(" ", "_")
                             .lower())
            formatted_val = (new_val.strip().replace("'", "").replace(" ", "_").lower())
            cleaned_rename_map[formatted_key] = formatted_val
        df = df.rename(columns=cleaned_rename_map)
        print("[INFO] Specific columns renamed via configuration map.")
    return df

def drop_columns(df, columns_to_drop=None):
    if df is None: return None
    df = df.copy()
    if columns_to_drop:
        if isinstance(columns_to_drop, str):
            target_cols = [c.strip().replace("'", "").replace(" ", "_").lower() for c in columns_to_drop.split(',')]
        else:
            target_cols = [c.strip().replace("'", "").replace(" ", "_").lower() for c in columns_to_drop]
        existing_cols = [c for c in target_cols if c in df.columns]
        if existing_cols:
            df = df.drop(columns=existing_cols)
            print(f"[INFO] Dropped {len(existing_cols)} columns.")
    return df

def merge_and_drop_other_columns(df, merge_pairs=None):
    if df is None: return None
    df = df.copy()
    if merge_pairs:
        count = 0
        for target, source in merge_pairs:
            fmt_target = target.strip().replace(" ", "_").lower()
            fmt_source = source.strip().replace(" ", "_").lower()
            if fmt_target in df.columns and fmt_source in df.columns:
                source_clean = df[fmt_source].replace(r'^\s*$', np.nan, regex=True)
                s_series = source_clean.fillna('').astype(str)
                has_source = (s_series != '')
                t_series = df[fmt_target].astype(str)
                pattern = r'other\s*\(please specify\)[,\s]*'
                t_series_cleaned = t_series.str.replace(pattern, '', case=False, regex=True)
                final_target = np.where(has_source, t_series_cleaned, t_series)
                final_target = pd.Series(final_target, index=df.index)
                separator = np.where((final_target != '') & (has_source), ', ', '')
                merged_col = final_target + separator + s_series
                df[fmt_target] = merged_col.replace(r'^\s*$', np.nan, regex=True)
                df = df.drop(columns=[fmt_source])
                count += 1
        print(f"[INFO] Merged and dropped {count} 'Other' columns.")
    return df

def process_smells_logic(text):
    if not isinstance(text, str): return text
    text = text.replace("sounds, textures", "sounds textures")
    match = re.search(r'(smells)(\s*)(.*)', text)
    if match:
        if match.group(3).strip():
            return text.replace(match.group(0), f"smells, {match.group(3).strip()}")
        return text.replace(match.group(0), "smells")
    return text

def standardise_text_data(df):
    if df is None: return None
    df = df.copy()
    str_cols = df.select_dtypes(include=['object']).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.lower().str.strip()
        df[col] = df[col].replace({'nan': np.nan, 'none': np.nan})
    if 'repetitive_behaviours' in df.columns:
        df['repetitive_behaviours'] = df['repetitive_behaviours'].apply(process_smells_logic)
    print("[INFO] Text data standardised.")
    return df

def sanitize_open_text_fields(df):
    """
    Cleans specific open-text fields for NLP analysis.
    - Removes 'nan' strings.
    - Removes special chars (*, #) and intermittent punctuation (:, ').
    - Keeps sentence punctuation (., !, ?).
    """
    if df is None: return None
    df = df.copy()

    # Define columns to clean
    target_columns = [
        'most_challenging_aspect',
        'solution_most_challenging_aspect',
        'reason_belief_ai'
    ]

    # Regex explanation:
    # [^\w\s.,!?] -> Match anything that is NOT a word char, space, or sentence punctuation (.,!?)
    # This effectively removes *, :, ', #, etc.
    clean_pattern = r'[^\w\s.,!?]'

    for col in target_columns:
        if col in df.columns:
            # 1. Convert to string and handle basic NaN
            df[col] = df[col].astype(str).fillna('')

            # 2. Remove the string "nan" specifically
            df[col] = df[col].replace('nan', '', regex=False)

            # 3. Apply regex cleaning (remove unwanted symbols)
            df[col] = df[col].apply(lambda x: re.sub(clean_pattern, '', x) if isinstance(x, str) else x)

            # 4. Clean up extra spaces created by removal
            df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else x)

            # 5. Restore NaN for empty strings (optional, depends on preference)
            df[col] = df[col].replace('', 'not specified')

    print("[INFO] Open text fields sanitized for NLP.")
    return df

def impute_missing_values(df):
    if df is None: return None
    df = df.copy()
    obj_cols = df.select_dtypes(include=['object']).columns
    df[obj_cols] = df[obj_cols].fillna('not specified')
    print("[INFO] Missing values imputed.")
    return df

def save_to_csv(df, output_path):
    if df is None: return None
    try:
        df.to_csv(output_path, index=False)
        print(f"[SUCCESS] File saved to: {output_path}")
        # Trigger download in Colab
        files.download(output_path)
        return df
    except Exception as e:
        print(f"[ERROR] Could not save file: {e}")
        return df

# ==========================================
# 3. PIPELINE ORCHESTRATOR
# ==========================================

def preprocess_pipeline(merge_pairs=None, drop_input=None, rename_map=None):
    """
    Main Pipeline Execution
    """
    print("\n--- Starting Data Preprocessing Pipeline ---")

    # 1. Load Data via Upload
    raw_df, original_filename = load_data_from_upload()

    if raw_df is None:
        print("Pipeline aborted.")
        return None

    print(f"[INFO] Input data shape: {raw_df.shape}") # Added this line

    # 2. Generate Dynamic Output Filename
    base_name = os.path.splitext(original_filename)[0]
    suffix = base_name[-8:] # Get last 8 chars
    final_output_path = f"survey_data_processed_{suffix}.csv"

    print(f"[INFO] Output will be saved as: {final_output_path}")

    # 3. Execute Processing Steps
    processed_df = (
        raw_df
        .pipe(format_column_names)
        .pipe(rename_specific_columns, rename_map=rename_map)
        .pipe(drop_columns, columns_to_drop=drop_input)
        .pipe(merge_and_drop_other_columns, merge_pairs=merge_pairs)
        .pipe(standardise_text_data)
        .pipe(sanitize_open_text_fields)
        .pipe(impute_missing_values)
        .pipe(save_to_csv, output_path=final_output_path)
    )

    if processed_df is not None:
        print(f"[INFO] Output data shape: {processed_df.shape}") # Added this line

    print("--- Data Preprocessing Complete ---")
    return processed_df

# ==========================================
# 4. EXECUTION BLOCK
# ==========================================

if __name__ == "__main__":
    # Run the pipeline (No arguments needed for file path as it asks for upload)
    preprocess_pipeline(
        merge_pairs=MERGE_CONFIG,
        drop_input=COLUMNS_TO_DROP,
        rename_map=RENAME_CONFIG
    )